# Table I — Per-subject decoding performance

Reproduces **Table I** of the paper:

> *Factors Influencing Speech Perception Decoding with fNIRS: Analysis of Spatial and Temporal Characteristics and Signal Complexity*  
> Posso-Murillo, Sanchez-Giraldo, Bae (2025)

For each of the 8 subjects, we run a **5-outer × 3-inner** stratified nested cross-validation,
with **ADASYN** class balancing on the training fold, on a flattened (channels × time) feature
vector (HbO and HbR concatenated). Two linear decoders are evaluated:

* **Linear SVC** — grid-searched over `C ∈ {1e-3, 1e-2, 1e-1, 1}`
* **LDA** — no hyperparameters

Output: mean ± std AUC across the 5 outer folds, per (subject, model). The notebook also
saves the SHAP values used by every figure script in the repo.

## Runtime

End-to-end this notebook takes **~15–25 minutes** on a modern laptop CPU.
Most of that is `final_audio_data` (preprocessing) plus SHAP value computation per fold.
The first run also downloads ~1 GB of fNIRS data via `mne_nirs.datasets.audio_or_visual_speech`.

## Outputs

* `outputs/auc/auc_per_subject.csv` — long-format AUC per (subject, model, fold)
* `outputs/shap/subject_*/shap_values_{svc,lda}_subject_*.npz` — SHAP values consumed by
  `fig_1_and_3.py`, `fig_2.py`, and `table_2.py`

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# Make the repo root importable when running the notebook from notebooks/
sys.path.insert(0, os.path.abspath('..'))
os.chdir(os.path.abspath('..'))
print('cwd:', os.getcwd())

In [ ]:
import numpy as np
import pandas as pd

from audio_data import final_audio_data
from train_shap import machine_learning_exp

## 1. Run the nested CV pipeline

Train SVC and LDA on each subject. Saves SHAP values and per-fold AUCs to `outputs/`.

In [ ]:
SUBJECTS = list(range(8))
MODELS = ('svc', 'lda')
T_MIN, T_MAX = 0.0, 18.0
OUT_ROOT = 'outputs'

os.makedirs(os.path.join(OUT_ROOT, 'auc'), exist_ok=True)
auc_csv = os.path.join(OUT_ROOT, 'auc', 'auc_per_subject.csv')

rows = []
for subject in SUBJECTS:
    print(f'\n=== Subject {subject} ===')
    X_arr, Y, _ = final_audio_data(subject, T_MIN, T_MAX, chroma=None)
    print(f'  X: {X_arr.shape}  (trials, channels, time)')
    X_flat = X_arr.reshape(X_arr.shape[0], -1)
    X = pd.DataFrame(X_flat)
    X.columns = X.columns.astype(str)

    shap_dir = os.path.join(OUT_ROOT, 'shap', f'subject_{subject}')
    os.makedirs(shap_dir, exist_ok=True)

    for model in MODELS:
        print(f'  Model: {model}')
        shap_values, hps, auc_per_fold, _ = machine_learning_exp(model, X, Y)
        np.savez(
            os.path.join(shap_dir, f'shap_values_{model}_subject_{subject}.npz'),
            folder_1=shap_values[0], folder_2=shap_values[1],
            folder_3=shap_values[2], folder_4=shap_values[3],
            folder_5=shap_values[4],
        )
        for fold, (auc, hp) in enumerate(zip(auc_per_fold, hps), start=1):
            rows.append({'subject': subject, 'model': model, 'fold': fold, 'AUC': auc, 'C': hp})
        mean = float(np.mean(auc_per_fold))
        std = float(np.std(auc_per_fold))
        print(f'    {model.upper()} AUC = {mean:.3f} +/- {std:.3f}')

pd.DataFrame(rows).to_csv(auc_csv, index=False)
print(f'\nWrote {auc_csv}')

## 2. Render Table I

Format the long-format AUC CSV into the table from the paper.

In [ ]:
df = pd.read_csv(auc_csv)
summary = (
    df.groupby(['model', 'subject'])
      .agg(mean=('AUC', 'mean'), std=('AUC', 'std'))
      .reset_index()
)
summary['cell'] = summary.apply(
    lambda r: f"{r['mean']:.2f} \u00b1 {r['std']:.2f}", axis=1
)
table_1 = summary.pivot(index='model', columns='subject', values='cell')
table_1.columns = [f'sub.{c}' for c in table_1.columns]
table_1.index = table_1.index.str.upper()
table_1 = table_1.loc[['SVC', 'LDA']]
print('Table I: Decoding performances')
table_1

In [ ]:
print(table_1.to_markdown())

## 3. Sanity check — ANOVA / Kruskal-Wallis between SVC and LDA

The paper reports no significant difference between SVC and LDA. We check that here.

In [ ]:
from scipy import stats
svc_aucs = df[df.model == 'svc']['AUC'].to_numpy()
lda_aucs = df[df.model == 'lda']['AUC'].to_numpy()
_, p_shap_svc = stats.shapiro(svc_aucs)
_, p_shap_lda = stats.shapiro(lda_aucs)
if p_shap_svc > 0.05 and p_shap_lda > 0.05:
    f, p = stats.f_oneway(svc_aucs, lda_aucs)
    print(f'ANOVA: F={f:.3f}, p={p:.3f}')
else:
    h, p = stats.kruskal(svc_aucs, lda_aucs)
    print(f'Kruskal-Wallis: H={h:.3f}, p={p:.3f}')

## Next

Generate the figures with:

```bash
python fig_1_and_3.py --model svc
python fig_1_and_3.py --model lda
python fig_2.py   --model svc
python fig_2.py   --model lda
python fig_4.py
python table_2.py --model svc
python table_2.py --model lda
```